In [1]:
import copy

import numpy

import cloudvolume
import kimimaro
import fastremap

np = numpy

In [2]:
import ac_pcg.label
import ac_pcg.chunks
import ac_pcg.skeletons

In [3]:
import gzip
import pathlib
import pickle

def read_gzip_array(fn, preprocess_func=lambda x: x):
    with gzip.open(fn, "rb") as f:
        a = numpy.load(f)
    return preprocess_func(a)

test_data_path = pathlib.Path(
    "/allen/programs/celltypes/workgroups/em-connectomics/russelt/pcg_axconn/test_data_strip/"
)

test_data_labeled_array_path = test_data_path / "H17_x55_S32_230412_Pos42.npy.gz"
test_skels_path = test_data_path / "H17_x55_S32_230412_Pos42.skels.pkl"

In [4]:
labeled_array = read_gzip_array(test_data_labeled_array_path)
with test_skels_path.open(mode="rb") as skels_fobj:
    label_skels = pickle.load(skels_fobj)

In [5]:
%%time
import rtree

skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}


p = rtree.index.Property()
p.dimension = 3
label_skel_idx = rtree.index.Index(
    ((sk_id, skel_bb.to_list(), label_skels[sk_id]) for sk_id, skel_bb in skel_id_to_bboxes.items()),
    properties=p
)

CPU times: user 4.11 s, sys: 55.7 ms, total: 4.17 s
Wall time: 4.17 s


In [ ]:
# TODO concurrent processing, serial vertex assignment.
#   serial chunk processing method below

In [21]:
import time

chunk_size = (128, 128, 128)
chunk_boxes = ac_pcg.chunks.iterate_chunk_slice_boxes(
    labeled_array.shape, chunk_size)

labeler = ac_pcg.label.ChunkLabeler()

output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

output_skels = copy.deepcopy(label_skels)

tic = time.time()
for chunk_num, chunk_box in enumerate(chunk_boxes):
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(res.object, chunk_contains_bb)
        for res in label_skel_idx.intersection(
            chunk_contains_bb.to_list(), objects=True
        )
    ))
    try:
        subvol_skels, subvol_skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue

    oversegmented_subvol_arr, oversegmented_subvol_skels = kimimaro.utility.oversegment(
        subvol_arr, subvol_skels, downsample=6, progress=False)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_subvol_arr
        )
    }
    relabeled_arr = fastremap.remap(oversegmented_subvol_arr, lbl_map)
    
    output_arr[chunk_box.bbox.to_slices()] = relabeled_arr[...]

    # map new indices to original skel vertices
    for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
        output_skel = output_skels[skel.id]
        skel.segments = fastremap.remap(skel.segments, lbl_map)
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments
    

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

90 0.33299946784973145
100 1.0048799514770508
120 3.422220230102539
130 4.999111652374268
140 6.412767648696899
150 7.942875862121582
160 9.389643669128418
170 10.882501125335693
180 12.675294160842896
190 14.614686012268066
200 16.167608737945557
210 17.853861808776855
220 19.833218097686768
230 21.585639476776123
240 23.34796905517578
250 25.22839069366455
260 26.958542108535767
270 29.363019466400146
280 32.04717516899109
290 34.52350926399231
300 37.17863392829895
310 39.763710498809814
320 41.678956031799316
330 43.540791273117065
340 45.170337200164795
350 46.742698431015015
360 48.87941074371338
370 51.11362385749817
390 54.804993629455566
400 57.149229764938354
420 61.071292877197266
430 62.9443724155426
450 66.90235090255737
460 69.01357388496399
480 72.65243434906006
490 74.39393901824951
500 76.01613211631775
510 77.64941835403442
520 79.3649685382843
530 80.87492823600769
540 82.74465751647949
550 84.49940228462219
560 85.91674017906189
570 87.33101081848145
580 88.86639285

In [ ]:
# write out protobuf edges and precomputed array

In [7]:
from pychunkedgraph.graph.edges import Edges


ModuleNotFoundError: No module named 'pychunkedgraph'

In [ ]:
# no cross-chunk edges in how these are processed


In [31]:
e = output_skel.segments[output_skel.edges]

In [39]:
%%timeit
edges_mask = (e[:, 0] != e[:, 1])
skel_edges = numpy.unique(
    numpy.sort(
        e[edges_mask], axis=1
    ),
    axis=0)


66.8 μs ± 526 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
